In [1]:
# Dans Colab : Run cette cellule en premier
!pip install nba_api tensorflow scikit-learn pandas matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 6.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import time
from nba_api.stats.endpoints import leaguegamelog, boxscoretraditionalv2
from nba_api.stats.static import teams

# On fetch les matchs de la saison 2023-24
from nba_api.stats.endpoints import LeagueGameLog

gamelog = LeagueGameLog(season='2023-24', season_type_all_star='Regular Season')
games_df = gamelog.get_data_frames()[0]

print(f"Matchs trouvés : {len(games_df)}")
print(games_df[['GAME_ID','TEAM_ABBREVIATION','PTS']].head(6))

Matchs trouvés : 2460
      GAME_ID TEAM_ABBREVIATION  PTS
0  0022300061               DEN  119
1  0022300062               GSW  104
2  0022300061               LAL  107
3  0022300062               PHX  108
4  0022300071               NOP  111
5  0022300070               CHI  104


In [3]:
# Sauvegarder immédiatement → pas besoin de re-fetcher si tu redémarres
games_df.to_csv('nba_games_2324.csv', index=False)
print("Sauvegardé !")

Sauvegardé !


In [4]:
games_df = pd.read_csv('nba_games_2324.csv')

In [5]:
unique_game_ids = games_df['GAME_ID'].unique()[:200]
print(f"On va fetcher {len(unique_game_ids)} matchs")

On va fetcher 200 matchs


In [6]:
first = unique_game_ids[0]
print(first)

22300061


In [7]:
import pandas as pd

# Remove the limit on the number of columns displayed
pd.set_option('display.max_columns', None)


In [8]:
from nba_api.stats.endpoints import boxscoresummaryv3

# The game_id must be a 10-digit string for the API to return data correctly
formatted_id = '00' + str(first)
box = boxscoresummaryv3.BoxScoreSummaryV3(game_id=formatted_id)
line_score = box.get_data_frames()[4]
display(line_score)

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,teamWins,teamLosses,period1Score,period2Score,period3Score,period4Score,score
0,0022300061,1610612743,Denver,Nuggets,DEN,nuggets,1,0,34,29,24,32,119
1,0022300061,1610612747,Los Angeles,Lakers,LAL,lakers,0,1,20,34,26,27,107


In [11]:
from nba_api.stats.endpoints import boxscoresummaryv3

records = []

for i, game_id in enumerate(unique_game_ids[:200]):
    try:
        formatted_id = '00' + str(game_id)
        box = boxscoresummaryv3.BoxScoreSummaryV3(game_id=formatted_id)
        line_score = box.get_data_frames()[4]

        for _, row in line_score.iterrows():
            records.append({
                'game_id': game_id,
                'team': row['teamTricode'],
                'Q1': row['period1Score'],
                'Q2': row['period2Score'],
                'Q3': row['period3Score'],
                'Q4': row['period4Score'],
                'FINAL': row['score']
            })

        if i % 20 == 0:
            print(f"  {i}/{200} matchs fetchés...")

        time.sleep(0.6)

    except Exception as e:
        print(f"Erreur sur {game_id}: {e}")
        continue

quarters_df = pd.DataFrame(records)
quarters_df.to_csv('nba_quarters_2324.csv', index=False)
print(f"\nDone ! {len(quarters_df)} lignes sauvegardées.")
quarters_df.head(6)

  0/200 matchs fetchés...
  20/200 matchs fetchés...
  40/200 matchs fetchés...
  60/200 matchs fetchés...
  80/200 matchs fetchés...
  100/200 matchs fetchés...
  120/200 matchs fetchés...
  140/200 matchs fetchés...
  160/200 matchs fetchés...
  180/200 matchs fetchés...

Done ! 400 lignes sauvegardées.


,game_id,team,Q1,Q2,Q3,Q4,FINAL
0,22300061,DEN,34,29,24,32,119
1,22300061,LAL,20,34,26,27,107
2,22300062,GSW,28,18,40,18,104
3,22300062,PHX,28,33,19,28,108
4,22300071,MEM,28,19,23,34,104
5,22300071,NOP,25,32,25,29,111


In [10]:
from nba_api.stats.endpoints import boxscoresummaryv3

# 1. Use the formatted 10-digit string ID
game_id = unique_game_ids[0]
formatted_id = '00' + str(game_id)

# 2. Fetch the data
box = boxscoresummaryv3.BoxScoreSummaryV3(game_id=formatted_id)

# 3. Access the LineScore DataFrame (usually index 4 in get_data_frames())
line_score_df = box.get_data_frames()[4]

print(f"DataFrame Shape: {line_score_df.shape}")
display(line_score_df.head())

DataFrame Shape: (2, 13)


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,teamWins,teamLosses,period1Score,period2Score,period3Score,period4Score,score
0,0022300061,1610612743,Denver,Nuggets,DEN,nuggets,1,0,34,29,24,32,119
1,0022300061,1610612747,Los Angeles,Lakers,LAL,lakers,0,1,20,34,26,27,107
